# SME Capital Matching: Data Cleaning

This notebook takes the raw application file from the `Datasets` folder, repairs the faults in it, and produces a table that is ready to be analysed.

The steps are as follows.

1. [Find the `Datasets` folder and load the raw data;](#step-1)
2. [Rename the raw survey columns to short, tidy names;](#step-2)
3. [Define the cleaning functions and lookup tables;](#step-3)
4. [Apply the cleaning functions to every field;](#step-4)
5. [Drop the repeated applications;](#step-5)
6. [Build and display the before and after summary table;](#step-6)
7. [Build a styled Excel workbook and save it to the `Datasets` folder.](#step-7)


| Phase | File | Folder |
|---|---|---|
| The raw and faulty extract | `capital_matching_applications.csv` (1,186 rows) | `Datasets` |
| ⬇ *this notebook reads and cleans it, then produces* | `Data_Cleaning.ipynb` | `Python_Notebooks` |
| The tidy table, ready to analyse | `Capital_Matching_Cleaned_Data.xlsx` | `Datasets` |
| ⬇ *the next notebook reads that file and scores it* | `Funding_Readiness_Segmentation.ipynb` | `Python_Notebooks` |
| **The final result**, the working data model | `Funding_Readiness_Segmentation.xlsx` | `Datasets` |

<a id="step-1"></a>
## Step 1: Find the `Datasets` Folder and Load the Raw Data

This step performs three actions.

**1. Locate the `Datasets` folder.** Search the current folder, then each folder above it, until the folder named `Datasets` is found.

**2. Set the file paths.** Point to the raw input file, `capital_matching_applications.csv`, and to the cleaned output file, `Capital_Matching_Cleaned_Data.xlsx`, which Step 7 saves back into the same folder.

**3. Load the raw data.** Read `capital_matching_applications.csv` with every column set to text. This prevents fields that look numeric, such as the money columns, from being converted automatically and corrupted before they can be cleaned. Reading the data as text preserves every original character for the steps that follow.

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path


# 1. Locate the Datasets folder.
def find_datasets_folder():
    """Look in this folder, then the one above it, and so on, until 'Datasets' is found."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "Datasets").is_dir():
            return folder / "Datasets"
    raise FileNotFoundError(
        "Could not find a folder named 'Datasets'. Please open this notebook from "
        "inside the project folder (the one that contains 'Datasets')."
    )

# 2. Set the file paths: the raw input file, and the cleaned output file saved in Step 7.
DATASETS = find_datasets_folder()
RAW_FILE = DATASETS / "capital_matching_applications.csv"
CLEAN_FILE = DATASETS / "Capital_Matching_Cleaned_Data.xlsx"

print(f"Datasets folder : {DATASETS}")
print(f"Reading from    : {RAW_FILE.name}")
print(f"Will save to    : {CLEAN_FILE.name}")

# 3. Load the raw data with every column set to text.
raw = pd.read_csv(RAW_FILE, dtype=str)
print(f"\nLoaded {raw.shape[0]:,} rows x {raw.shape[1]} columns")

# 4. Preview the fields that carry the data-quality problems this notebook exists to fix.
#    Identifying fields are deliberately kept out of every preview in this notebook:
#    business and contact names, email addresses, phone numbers, registration numbers
#    and street addresses. They are loaded and cleaned like any other column, but they
#    are never printed, because this same code runs unchanged against a confidential
#    export where displaying them would write them into a saved output.
PREVIEW_COLUMNS = [
    "Created",
    "Industry",
    "Province",
    "City/Town",
    "Funding Ask (Amount):",
    "Annual Revenue Range 2024",
    "Actual Revenue 2025",
    "Existing number of employees ",
    "BEE Level",
]
raw[PREVIEW_COLUMNS].head(5)

Datasets folder : C:\Users\IC Clearwater\OneDrive\Documents\GitHub\SME_Capital_Funding_Optimization\Datasets
Reading from    : capital_matching_applications.csv
Will save to    : Capital_Matching_Cleaned_Data.xlsx

Loaded 1,186 rows x 27 columns


,Created,Industry,Province,City/Town,Funding Ask (Amount):,Annual Revenue Range 2024,Actual Revenue 2025,Existing number of employees,BEE Level
0,2025-08-15 19:19:12,Education,Gauteng,Kuruman,600000_________,R501K to R1M,603752_________,1.0,Level 1
1,2025-08-25 07:30:21,NaN,Gauteng,Richards Bay,1190000________,0 to R500K,298746_________,5.0,Level 1
2,2025-08-14 18:10:50,Transportation and Logistics,Western Cape,Centurion,3450000________,R1 000 001 to R5M,2297089________,3.0,Level 3
3,2025-07-09 22:54:36,Manufacturing,Gauteng,Worcester,350000_________,0 to R500K,70549__________,1.0,Level 1
4,2025-08-14 05:29:34,Waste Management,Gauteng,Kuruman,3300000________,R1 000 001 to R5M,1098973________,3.0,Level 1


<a id="step-2"></a>
## Step 2: Rename the Raw Survey Columns to Short, Tidy Names

The raw headings are long and inconsistent, so each one is mapped to a shorter and more legible name. The full mapping below is the data dictionary, which records where each tidy name comes from in the raw dataset.

In [2]:
# 1. Data dictionary: map each raw survey column to a shorter tidy name.
COLUMN_RENAME = {
    "Created": "created_at",
    "Name:: Name & Surname": "applicant_name",
    "Company Name:": "company_name",
    "Company Registration Number:": "company_reg_no",
    "Industry": "industry",
    "Province": "province",
    "Location of Head Office (address)": "head_office_address",
    "Funding Ask (Amount):": "funding_ask_raw",
    "City/Town": "city_town",
    "Type of Funding": "funding_type",
    "Funding Requirements (what is the funding required for).": "funding_purpose",
    "Annual Revenue Range 2023": "revenue_band_2023",
    "Annual Revenue Range 2024": "revenue_band_2024",
    "Actual Revenue 2025": "revenue_2025_raw",
    "Company Overview ": "company_overview",
    "Existing number of employees ": "employees_raw",
    "Increase of jobs anticipated through the capital access": "jobs_anticipated_raw",
    "Number of Shareholders": "shareholders_raw",
    "What percentage of your business is owned by individuals aged 18 to 35? (Please select one)": "youth_ownership_band",
    "What percentage of your business is owned by women? (Please select one)": "women_ownership_band",
    "What percentage of your business is under Black ownership? ": "black_ownership_band",
    "BEE Level": "bee_level_raw",
    "Area": "area",
    "What percentage of your business is owned by individuals living with a disability? (Please select one)": "disability_ownership_band",
    "Contact person Name and Surname": "contact_name",
    "Email address:": "email",
    "Contact number:": "contact_number",
}

# 2. Apply the renaming to the raw data.
df = raw.rename(columns=COLUMN_RENAME).copy()

# 3. Display the data dictionary as a legible table.
data_dictionary = pd.DataFrame(
    [(orig, short) for orig, short in COLUMN_RENAME.items()],
    columns=["Original survey column", "Tidy name"],
)
display(data_dictionary)

,Original survey column,Tidy name
0,Created,created_at
1,Name:: Name & Surname,applicant_name
2,Company Name:,company_name
3,Company Registration Number:,company_reg_no
4,Industry,industry
5,Province,province
6,Location of Head Office (address),head_office_address
7,Funding Ask (Amount):,funding_ask_raw
8,City/Town,city_town
9,Type of Funding,funding_type


<a id="step-3"></a>
## Step 3: Define the Cleaning Functions and Lookup Tables

This step defines the small functions and the lookup tables used to clean each field, where each function performs a single simple task.

1. **`strip_text`**: removes surrounding spaces, collapses repeated spaces, and turns empty or `"null"` values into a true blank.

2. **`parse_money`**: converts a malformed money entry, such as `250000_________` or `R 1,850,000`, into a clean number (`250000` or `1850000`).

3. **`parse_percent_band`**: converts an ownership percentage range, such as `50–74%`, into one representative value taken from the middle of that range (62). Both the long dash (`–`) and the ordinary hyphen (`-`) appear in the raw data, and the function treats them as the same character.

4. **`parse_bee_level`**: extracts the number from a value such as `"Level 1"`. B-BBEE stands for Broad-Based Black Economic Empowerment, which is the South African framework that measures how far a business has moved towards Black ownership and inclusion, and **Level 1 is the strongest rating**.

5. **`fix_province`**: maps spelling variants, such as `kzn` and `KwaZulu Natal`, to a single standard name.

6. **`parse_count`**: converts a count into a number, reading counts written out in words (`one`, `Myself`) as the figure they state. Values that answer a different question, such as an ownership percentage or a company name, cannot be interpreted as a count and are left blank.

This step also defines the lookup tables used for the revenue bands, which are an ordered scale from 0 to 6 and a matching Rand value for the middle of each band.

In [3]:
# Revenue text bands mapped to an ordered scale (0 = smallest ... 6 = largest)
REVENUE_BAND_ORDER = {
    "0 to R500K": 0, 
    "R501K to R1M": 1, 
    "R1 000 001 to R5M": 2, 
    "R5 000 001 to R10M": 3,
    "R10 000 001 to R20M": 4, 
    "R20 000 001 to R50M": 5, 
    "Above R50M": 6,
}

# A representative Rand midpoint for each revenue band, used for sizing
REVENUE_BAND_MIDPOINT = {
    "0 to R500K": 250_000, 
    "R501K to R1M": 750_000, 
    "R1 000 001 to R5M": 3_000_000,
    "R5 000 001 to R10M": 7_500_000, 
    "R10 000 001 to R20M": 15_000_000,
    "R20 000 001 to R50M": 35_000_000, 
    "Above R50M": 75_000_000,
}

# Ownership percentage bands mapped to a representative midpoint percentage
PERCENT_BAND_MIDPOINT = {
    "0%": 0.0, 
    "1–24%": 12.5, 
    "25–49%": 37.0, 
    "50–74%": 62.0, 
    "75–99%": 87.0, 
    "100%": 100.0
    }

# Province spelling variants mapped to one standard spelling
PROVINCE_FIX = {
    "kzn": "KwaZulu-Natal", "kwazulu natal": "KwaZulu-Natal", "kwazulu-natal": "KwaZulu-Natal",
    "gauteng": "Gauteng", "limpopo": "Limpopo", 
    "western cape": "Western Cape",
    "eastern cape": "Eastern Cape", 
    "northern cape": "Northern Cape", 
    "north west": "North West",
    "free state": "Free State", 
    "mpumalanga": "Mpumalanga",
}

# Counts written out in words, mapped to the number they state
WORDED_COUNT = {
    "one": 1.0,
    "myself": 1.0,
    "sole": 1.0,
    "sole owner": 1.0,
}

# 1. strip_text
def strip_text(x):
    if pd.isna(x):
        return np.nan
    s = re.sub(r"\s+", " ", str(x)).strip()
    return np.nan if s == "" or s.lower() == "nan" else s

# 2. parse_money
def parse_money(x):
    if pd.isna(x):
        return np.nan
    s = re.sub(r"[^0-9.]", "", str(x))   
    s = re.sub(r"\.(?=.*\.)", "", s)     
    if s in ("", "."):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan
    
# 3. parse_percent_band
def parse_percent_band(x): 
    if pd.isna(x):
        return np.nan
    return PERCENT_BAND_MIDPOINT.get(str(x).strip().replace("-", "–"), np.nan)

# 4. parse_bee_level
def parse_bee_level(x):
    if pd.isna(x):
        return np.nan
    m = re.search(r"(\d+)", str(x))
    return int(m.group(1)) if m else np.nan

# 5. fix_province
def fix_province(x):
    if pd.isna(x):
        return np.nan
    return PROVINCE_FIX.get(str(x).strip().lower(), str(x).strip())

# 6. parse_count
def parse_count(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s.lower() in WORDED_COUNT:
        return WORDED_COUNT[s.lower()]
    return pd.to_numeric(s, errors="coerce")

# Confirm every utility function and lookup table defined above is ready for use.
utility_functions = [strip_text, parse_money, parse_percent_band, parse_bee_level, fix_province, parse_count]
lookup_tables = [REVENUE_BAND_ORDER, REVENUE_BAND_MIDPOINT, PERCENT_BAND_MIDPOINT, PROVINCE_FIX, WORDED_COUNT]

print(f"Utility functions ready : {len(utility_functions)}")
print(f"Lookup tables ready     : {len(lookup_tables)}")

Utility functions ready : 6
Lookup tables ready     : 5


<a id="step-4"></a>
## Step 4: Apply the Cleaning Functions to Every Field

This step applies the functions defined in Step 3 to every field, one group of fields at a time, in the following order.

1. Trim the spaces from every text column.
2. Parse the timestamp field into a real date and time value.
3. Standardise the province spellings and capitalise the city names, so that `johannesburg` and `Johannesburg ` resolve to the same value.
4. Parse the money fields, then cap the impossible values: anything above R2 billion is treated as a typing error and blanked out.
5. Convert both revenue range columns into the ordered 0 to 6 scale and a matching Rand value.
6. Convert the employee, job and shareholder counts into numbers.
7. Convert the four ownership percentage ranges into single representative numbers.
8. Extract the B-BBEE level number.
9. Convert email addresses to lower case, so that `Jo@X.com` and `jo@x.com` are recognised as the same address.

In [4]:
# 1. Trim whitespace from every text column.
for c in df.columns:
    df[c] = df[c].map(strip_text)

# 2. Parse the timestamp field into a real date and time value.
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

# 3. Standardise province spellings and title-case city names, so that `johannesburg` and `Johannesburg ` resolve to the same value.
df["province"] = df["province"].map(fix_province)
df["industry"] = df["industry"].str.strip()
df["city_town"] = df["city_town"].str.title()

# 4. Parse the money fields, then cap impossible outliers: any value above R2 billion is treated as a data-entry error and blanked out.
df["funding_ask_zar"] = df["funding_ask_raw"].map(parse_money)
df["revenue_2025_zar"] = df["revenue_2025_raw"].map(parse_money)

OUTLIER_CAP = 2_000_000_000
outliers_removed = 0
for col in ["funding_ask_zar", "revenue_2025_zar"]:
    n = int((df[col] > OUTLIER_CAP).sum())
    outliers_removed += n
    df.loc[df[col] > OUTLIER_CAP, col] = np.nan

# 5. Convert both revenue-range columns into the ordered 0–6 scale and a corresponding Rand midpoint.
for yr in ["2023", "2024"]:
    band = f"revenue_band_{yr}"
    df[f"revenue_scale_{yr}"] = df[band].map(REVENUE_BAND_ORDER)
    df[f"revenue_mid_{yr}"] = df[band].map(REVENUE_BAND_MIDPOINT)

# 6. Convert the employee, job and shareholder counts into numbers. The shareholder
#    column also holds counts written out in words, which parse_count reads as figures.
df["employees"] = pd.to_numeric(df["employees_raw"], errors="coerce")
df["jobs_anticipated"] = pd.to_numeric(df["jobs_anticipated_raw"], errors="coerce")
df["shareholders"] = df["shareholders_raw"].map(parse_count)

# 7. Convert the four ownership-percentage ranges into midpoint numbers.
df["youth_ownership_pct"] = df["youth_ownership_band"].map(parse_percent_band)
df["women_ownership_pct"] = df["women_ownership_band"].map(parse_percent_band)
df["black_ownership_pct"] = df["black_ownership_band"].map(parse_percent_band)
df["disability_ownership_pct"] = df["disability_ownership_band"].map(parse_percent_band)

# 8. Extract the BEE level number.
df["bee_level"] = df["bee_level_raw"].map(parse_bee_level)

# 9. Convert email addresses to lower case, so that `Jo@X.com` and `jo@x.com` are recognised as the same address.
df["email"] = df["email"].str.lower()

# Safety check: confirm the whitespace trim was applied. If this count is not 0,
# the cleaning has not worked and the figures that follow cannot be trusted.
stray_spaces = 0
for c in df.columns:
    stray_spaces += int(df[c].map(lambda v: isinstance(v, str) and v != v.strip()).sum())

print(f"Cleaning done. Impossible (> R2bn) money values blanked out: {outliers_removed}")
print(f"Shareholder counts written in words, read as figures: {int(df['shareholders_raw'].map(lambda v: isinstance(v, str) and v.strip().lower() in WORDED_COUNT).sum())}")
print(f"Text values still carrying stray spaces (must be 0): {stray_spaces}")
print("\nA sample of cleaned columns for review (identifying fields excluded, as in Step 1):")
df[["industry", "province", "city_town", "funding_ask_zar", "revenue_2025_zar",
    "revenue_scale_2024", "bee_level", "black_ownership_pct"]].head(6)

Cleaning done. Impossible (> R2bn) money values blanked out: 12
Shareholder counts written in words, read as figures: 12
Text values still carrying stray spaces (must be 0): 0

A sample of cleaned columns for review (identifying fields excluded, as in Step 1):


,industry,province,city_town,funding_ask_zar,revenue_2025_zar,revenue_scale_2024,bee_level,black_ownership_pct
0,Education,Gauteng,Kuruman,600000.0,603752.0,1,1,100.0
1,NaN,Gauteng,Richards Bay,1190000.0,298746.0,0,1,100.0
2,Transportation and Logistics,Western Cape,Centurion,3450000.0,2297089.0,2,3,100.0
3,Manufacturing,Gauteng,Worcester,350000.0,70549.0,0,1,100.0
4,Waste Management,Gauteng,Kuruman,3300000.0,1098973.0,2,1,100.0
5,Information & Communication Technology,Western Cape,Ladysmith,30000.0,0.0,0,1,100.0


<a id="step-5"></a>
## Step 5: Drop the Repeated Applications

This step removes the applications that appear more than once in the raw export, in two passes, keeping the first copy each time.

1. **Remove the exact duplicates.** Drop rows that are identical in every field.
2. **Remove the near duplicates.** Drop rows that share the same company name, email address and submission time. This removes the same application submitted twice with a minor difference somewhere else.

The counts before and after are printed, so that the reduction in rows is fully visible.

In [5]:
# Record the starting row count.
rows_in = len(df)

# 1. Remove exact duplicates: rows identical in every field, keeping the first copy.
df = df.drop_duplicates()
after_exact = len(df)

# 2. Remove near-duplicates: rows sharing the same company name, email address and submission time.
df = df.drop_duplicates(subset=["company_name", "email", "created_at"], keep="first").reset_index(drop=True)
rows_out = len(df)

# Print the before-and-after counts.
print(f"Rows in                     : {rows_in:,}")
print(f"After removing exact copies : {after_exact:,}  (-{rows_in - after_exact})")
print(f"After removing near-copies  : {rows_out:,}  (-{after_exact - rows_out})")
print(f"Unique businesses remaining : {rows_out:,}")

Rows in                     : 1,186
After removing exact copies : 1,118  (-68)
After removing near-copies  : 1,116  (-2)
Unique businesses remaining : 1,116


<a id="step-6"></a>
## Step 6: Build and Display the Before and After Summary Table

This step builds a side-by-side summary of the raw data against the cleaned data, so that the effect of the cleaning can be seen in one table. It gives the number of rows at the start and at the end, and the number of malformed money values recovered into usable numbers.

In [6]:
# 1. Count how many values in a column are already usable numbers.
def clean_number_count(series):
    return int(series.map(lambda x: bool(re.fullmatch(r"\d+(\.\d+)?", str(x).strip())) if pd.notna(x) else False).sum())

# 2. Build and display the before-and-after summary table.
summary = pd.DataFrame(
    [
        ("Rows (applications)", f"{rows_in:,}", f"{rows_out:,}"),
        ("Funding ask: usable numbers", f"{clean_number_count(raw['Funding Ask (Amount):']):,}", f"{int(df['funding_ask_zar'].notna().sum()):,}"),
        ("2025 revenue: usable numbers", f"{clean_number_count(raw['Actual Revenue 2025']):,}", f"{int(df['revenue_2025_zar'].notna().sum()):,}"),
        ("Distinct province spellings", f"{raw['Province'].str.strip().nunique():,}", f"{df['province'].nunique():,}"),
        ("BEE level as a number", "0", f"{int(df['bee_level'].notna().sum()):,}"),
    ],
    columns=["Measure", "Before (raw)", "After (clean)"],
)
display(summary)
print("The underscore-padded money fields are now real numbers; provinces are standardised; "
      "duplicates are gone. The data is ready to score.")

,Measure,Before (raw),After (clean)
0,Rows (applications),"1,186","1,116"
1,Funding ask: usable numbers,10,"1,109"
2,2025 revenue: usable numbers,0,"1,112"
3,Distinct province spellings,9,9
4,BEE level as a number,0,"1,116"


The underscore-padded money fields are now real numbers; provinces are standardised; duplicates are gone. The data is ready to score.


<a id="step-7"></a>
## Step 7: Build a Styled Excel Workbook and Save It to the `Datasets` Folder

This step saves the cleaned table as a presentable Excel file, `Capital_Matching_Cleaned_Data.xlsx`, back into the same `Datasets` folder that the raw CSV file was read from.

The workbook is deliberately simple, holding one sheet with one table of the cleaned data, a styled heading row, and a frozen top row for easy scrolling. It contains no summaries and no extra sheets, because its only purpose is to pass the clean dataset to the next notebook.

**This file is the handover.** The `Funding_Readiness_Segmentation` notebook opens exactly this file, from exactly this folder, and uses it to score every business. Nothing else passes between the two notebooks, so if this file saves correctly, the next notebook will run.

In [7]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# 1. Define a helper that strips the invisible control characters Excel refuses to store.
#    Normal text and emoji are kept.
def clean_cell(v):
    if not isinstance(v, str):
        return v
    return "".join(
        ch for ch in v
        if ch in "\t\n\r" or (
            ord(ch) >= 0x20 and ord(ch) != 0x7f
            and not (0x80 <= ord(ch) <= 0x9f)
            and not (0xFDD0 <= ord(ch) <= 0xFDEF)
            and (ord(ch) & 0xFFFF) not in (0xFFFE, 0xFFFF)
        )
    )

# 2. Set the workbook styling: font, colours and cell borders.
NAVY="1F3864"; FONT="Arial"
thin=Side(style="thin",color="D9D9D9"); BORDER=Border(left=thin,right=thin,top=thin,bottom=thin)

# 3. Create the workbook and write the styled header row.
wb=Workbook(); ws=wb.active; ws.title="Clean Data"; ws.sheet_view.showGridLines=False
for j,col in enumerate(df.columns,1):
    cell=ws.cell(row=1,column=j,value=str(col))
    cell.font=Font(name=FONT,bold=True,color="FFFFFF",size=10)
    cell.fill=PatternFill("solid",fgColor=NAVY)
    cell.alignment=Alignment(horizontal="center",vertical="center",wrap_text=True)
    cell.border=BORDER

# 4. Write the data rows, applying a money format to the monetary columns.
money_cols={"funding_ask_zar","revenue_2025_zar","revenue_mid_2023","revenue_mid_2024"}
for i,(_,r) in enumerate(df.iterrows(),2):
    for j,col in enumerate(df.columns,1):
        v=r[col]
        if pd.isna(v): v=None
        elif isinstance(v,(np.integer,)): v=int(v)
        elif isinstance(v,(np.floating,)): v=float(v)
        cell=ws.cell(row=i,column=j,value=clean_cell(v))
        cell.font=Font(name=FONT,size=9); cell.border=BORDER; cell.alignment=Alignment(vertical="center")
        if col in money_cols: cell.number_format='#,##0;(#,##0);-'

# 5. Freeze the header row and set sensible column widths.
ws.freeze_panes="A2"
for c in ws.columns:
    letter=get_column_letter(c[0].column)
    length=max((len(str(cell.value)) if cell.value is not None else 0) for cell in c)
    ws.column_dimensions[letter].width=min(max(length+2,10),40)

# 6. Save the workbook to the Datasets folder, using the path set in Step 1.
wb.save(CLEAN_FILE)
print(f"Saved the clean data table -> {CLEAN_FILE}")
print(f"({df.shape[0]:,} rows x {df.shape[1]} columns)")
print("\nThe Funding_Readiness_Segmentation notebook now reads this exact file.")

Saved the clean data table -> C:\Users\IC Clearwater\OneDrive\Documents\GitHub\SME_Capital_Funding_Optimization\Datasets\Capital_Matching_Cleaned_Data.xlsx
(1,116 rows x 41 columns)

The Funding_Readiness_Segmentation notebook now reads this exact file.
